## 3. Download data (public, no personal Drive needed)

Images and labels are hosted as public Google Drive files and fetched with
`gdown`. Anyone running this notebook (including the instructor) gets the same
data without mounting their own Drive.

## 1. GPU check

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Clone code from GitHub

Pulls the latest code. Safe to re-run: clones if absent, otherwise `git pull`.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/Shuhrat-1/image2biomass.git"
PROJECT = Path("/content/image2biomass")

if PROJECT.exists():
    !cd {PROJECT} && git pull
else:
    !git clone {REPO_URL} {PROJECT}

print("contents:", os.listdir(PROJECT))

## 3. Download data (public, no personal Drive needed)

Images and labels are hosted as public Google Drive files and fetched with
`gdown`. Anyone running this notebook (including the instructor) gets the same
data without mounting their own Drive.

In [ ]:
import zipfile
from pathlib import Path

!pip install -q gdown

# Public Google Drive file IDs (Anyone with the link -> Viewer)
ZIP_FILE_ID = "1x2OaQ1GUIk2fwmFfXBcMg0SsmlN7ZwxB"   # labelled.zip
CSV_FILE_ID = "1s5y4JAp9x7lDKl4gFHmuRczDIWi-5l3u"   # labelled.csv

RAW = PROJECT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)
(PROJECT / "data" / "processed").mkdir(parents=True, exist_ok=True)

# CSV: prefer the copy already in the repo; otherwise download it.
if not (RAW / "labelled.csv").exists():
    import gdown
    gdown.download(id=CSV_FILE_ID, output=str(RAW / "labelled.csv"), quiet=False)

# Images: download the zip and extract to local disk (fast I/O).
if not (RAW / "labelled").exists():
    import gdown
    zip_path = PROJECT / "labelled.zip"
    gdown.download(id=ZIP_FILE_ID, output=str(zip_path), quiet=False)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(RAW)

n_jpg = len(list((RAW / "labelled").glob("*.jpg")))
print("labelled images:", n_jpg)
print("labelled.csv exists:", (RAW / "labelled.csv").exists())

If `labelled images: 0`, the zip contains a nested folder. Inspect with
`!ls /content/image2biomass/data/raw` and move the .jpg files so they sit
directly under `data/raw/labelled/`.

## 4. Import project

In [ ]:
import sys, importlib
sys.path.insert(0, str(PROJECT / 'src'))

import config, eda, splits, dataset, model, train
for m in (config, eda, splits, dataset, model, train):
    importlib.reload(m)

print("in_colab:", config.in_colab())
print("PROJECT_ROOT:", config.PROJECT_ROOT)
print("LABELLED_CSV exists:", config.LABELLED_CSV.exists())
print("device:", train.get_device())

## 5. Load data and smoke test

Quick 5-epoch run to confirm the GPU pipeline works before the full sweep.

In [ ]:
wide = eda.group_rare_species(eda.to_wide(eda.load_long()))
print("wide:", wide.shape)

smoke = train.cross_validate(
    wide, kind="resnet18",
    model_kwargs={"pretrained": True, "freeze_backbone": True},
    n_splits=5, batch_size=32, num_workers=2,
    train_kwargs={"max_epochs": 5, "patience": 3},
    verbose=True,
)
print("smoke weighted RMSE:", round(smoke["weighted_rmse_mean"], 3))

## 6. Full training — 3 models x 5 folds

- **simple_cnn**: from scratch (session 10)
- **resnet18 frozen**: feature extraction (session 12)
- **resnet18 fine-tune**: full fine-tuning (session 12), smaller LR

In [ ]:
common = dict(n_splits=5, img_size=224, batch_size=32, num_workers=2, verbose=True)
results = {}

results["simple_cnn"] = train.cross_validate(
    wide, kind="simple_cnn", model_kwargs={},
    train_kwargs={"max_epochs": 60, "patience": 10, "lr": 1e-3}, **common)

results["resnet18_frozen"] = train.cross_validate(
    wide, kind="resnet18",
    model_kwargs={"pretrained": True, "freeze_backbone": True},
    train_kwargs={"max_epochs": 40, "patience": 8, "lr": 1e-3}, **common)

results["resnet18_finetune"] = train.cross_validate(
    wide, kind="resnet18",
    model_kwargs={"pretrained": True, "freeze_backbone": False},
    train_kwargs={"max_epochs": 30, "patience": 8, "lr": 1e-4}, **common)

for name, r in results.items():
    print(f"{name:22} weighted RMSE = {r['weighted_rmse_mean']:.3f} "
          f"+/- {r['weighted_rmse_std']:.3f}")

## 7. Comparison with the tabular baseline

In [ ]:
import pandas as pd
rows = [
    {"model": "naive_median (tabular)", "weighted_rmse": 24.117},
    {"model": "RandomForest (tabular)", "weighted_rmse": 11.828},
]
for name, r in results.items():
    rows.append({"model": name + " (image)",
                 "weighted_rmse": round(r["weighted_rmse_mean"], 3)})
comparison = pd.DataFrame(rows).sort_values("weighted_rmse").reset_index(drop=True)
comparison

In [ ]:
best_name = min(results, key=lambda k: results[k]["weighted_rmse_mean"])
print("Best image model:", best_name)
results[best_name]["summary"]

## 8. Save results and a deployable checkpoint

Saves metrics to Drive (survive disconnects) and trains a final image-only model
on (almost) all data for the Gradio app.

In [ ]:
import json
OUTPUTS = PROJECT / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)

out = {name: {"weighted_rmse_mean": r["weighted_rmse_mean"],
              "weighted_rmse_std": r["weighted_rmse_std"]}
       for name, r in results.items()}
with open(OUTPUTS / "cnn_results.json", "w") as f:
    json.dump(out, f, indent=2)
comparison.to_csv(OUTPUTS / "model_comparison.csv", index=False)
print("metrics saved to", OUTPUTS)
print("Download from the Colab file browser (folder icon, left panel).")

In [ ]:
# Final deployable model (image-only fine-tuned ResNet) for the Gradio app
OUTPUTS = PROJECT / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)

final_model, info = train.train_final_model(
    wide, kind="resnet18",
    model_kwargs={"pretrained": True, "freeze_backbone": False},
    train_kwargs={"max_epochs": 30, "patience": 8, "lr": 1e-4},
)
ckpt_path = OUTPUTS / "best_model.pt"
train.save_checkpoint(final_model, info, ckpt_path)
print("checkpoint saved to", ckpt_path)
print("val weighted RMSE:", round(info["val_weighted_rmse"], 3))
print("Download best_model.pt from the Colab file browser for the Gradio app.")

## Notes / conclusions

- RandomForest on metadata (11.83) beats the best image model (fine-tune ~17.6):
  cheap ground measurements outperform imagery on 357 samples.
- Frozen ResNet (~24.7) < naive median (24.1): unadapted ImageNet features fail;
  fine-tuning is essential.
- Download `best_model.pt` from Drive for the Gradio app (see DEPLOY.md).